# lssem3d CuPy backend on an A100

Validated already on an NVIDIA DGX Spark (GB10) in Docker — the whole validation
ladder passes there (`CUPY_BACKEND.md`). What the Spark **cannot** answer is
performance: its FP64 rate is ~1/41 of its FP32 (measured 0.21 against 8.61
TFLOP/s) and its bandwidth measured 112 GB/s, *below* an M3 Max. This solver is
float64 and bandwidth-bound, so a production number has to come from
FP64-capable hardware.

**Runtime → Change runtime type → A100 GPU** before running. If you get an L4 or
T4 instead, cell 1 says so — FP64 numbers from those are not worth recording.

In [ ]:
# 1. What did we actually get?  (Colab does not always give you the A100.)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print('\nGPU:', name)
if 'A100' not in name:
    print('!! NOT an A100 -- FP64 numbers from this device are not comparable.')
    print('   Runtime -> Change runtime type -> A100, then Restart runtime.')

In [ ]:
# 2. FP64 sanity: the number that decides whether this hardware is usable.
#    DGX Spark GB10 reference: FP64 0.21, FP32 8.61 TFLOP/s, 112 GB/s.
#    An A100 should be ~9.7 TFLOP/s FP64 and 1.5-1.9 TB/s.
import cupy as cp, time
def timeit(f, n=5):
    f(); cp.cuda.Stream.null.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): f()
    cp.cuda.Stream.null.synchronize()
    return (time.perf_counter()-t0)/n
N = 4096
a = cp.random.rand(N,N,dtype=cp.float64); b = cp.random.rand(N,N,dtype=cp.float64)
print(f'FP64 GEMM  {2*N**3/timeit(lambda: a@b)/1e12:6.2f} TFLOP/s')
a32, b32 = a.astype(cp.float32), b.astype(cp.float32)
print(f'FP32 GEMM  {2*N**3/timeit(lambda: a32@b32)/1e12:6.2f} TFLOP/s')
n = 1 << 26
x = cp.random.rand(n,dtype=cp.float64); y = cp.random.rand(n,dtype=cp.float64); z = cp.empty_like(x)
print(f'bandwidth  {3*n*8/timeit(lambda: cp.add(x,y,out=z))/1e9:6.0f} GB/s (fp64 triad)')

In [ ]:
# 3. Get the code.  Public repo; the CuPy work is on its own branch.
#    Idempotent: clones the first time, pulls afterwards, so re-running this
#    cell picks up new commits instead of failing on an existing directory.
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin cupy-backend && git reset -q --hard origin/cupy-backend
else:
    !git clone -q --branch cupy-backend https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q matplotlib
import cupy, numpy, scipy
print('cupy', cupy.__version__, '| numpy', numpy.__version__, '| scipy', scipy.__version__)

## Validation first, performance second

A backend is not trusted until it re-passes the ladder — symmetry and self-parity
tests cannot find a consistently wrong operator. These gates are checked against
analytic values, so they should give the *same numbers* here as on the Spark.

In [ ]:
# 4. Operator parity against the NumPy reference (fast).
!python scratch/cupy_parity.py

In [ ]:
# 5. Gate 3 (balance) and Gate 1 (Stokes sigma vs the analytic 9.3137399).
#    Spark results to match:
#      sigma  9.3153041 / 9.3141300 / 9.3138373  at dt = 0.01/0.005/0.0025
#      balance worst deviation 6.65e-06
!python scratch/cupy_validation_ladder.py 3
!python scratch/cupy_validation_ladder.py 1

In [ ]:
# 6. Gate 2 -- the expensive one (z-convection order at tol 1e-12).
#    GB10 took 147 / 305 / 633 s per dt.  Expect much faster here.
#    Expected: 5.724e-07 / 1.431e-07 / 3.577e-08, order 2.00.
!python scratch/cupy_validation_ladder.py 2

In [ ]:
# 7. Production-scale throughput.
#
#    FIRST RUN ON THIS A100 gave a FLAT 11.45 ms at every size from 0.53 M to
#    6.17 M dof -- a 12x range in work with no change in wall clock.  That is
#    the signature of a DISPATCH-bound loop: the host cannot issue the kernels
#    faster than the GPU retires them, so the A100 idles.  Inferred real work
#    at 6.17 M dof is ~1.85 ms (2.5 GB of traffic at the 1356 GB/s measured in
#    cell 2), i.e. ~84% of the wall clock was the host talking.
#
#    Two fixes have since landed (commit 7d49462): a per-matvec host
#    SYNCHRONISATION removed from the gather-scatter, and the derivative calls
#    batched (14 einsums -> 2, and 8 -> 2 in the adjoint).  Measured dispatch
#    floor: 3.54 -> 1.09 ms, a 3.25x cut.
#
#    WHAT TO LOOK FOR NOW: the flat line should drop to roughly 3-4 ms, and
#    the largest cases should begin to SEPARATE from the smallest -- separation
#    is the sign that real GPU work is finally visible above the host cost.
!python scratch/bench_backends.py numpy cupy

In [ ]:
# 7b. WHERE the remaining host time goes.
#
#     At a deliberately tiny size the GPU work is negligible, so the wall clock
#     IS the dispatch cost -- and the breakdown says what to optimise next.
#     Reference (DGX Spark GB10, after the fixes): total 1.088 ms, of which
#     apply_L 0.484, apply_LT 0.503, gather-scatter 0.085.
#
#     If the total here is much larger than ~1 ms, this host is slower at
#     issuing work than the Spark's Grace cores -- which is exactly why the
#     benchmark above was flat in the first place.
!python scratch/cupy_dispatch_profile.py

### Why not CUDA graphs?

Graph capture is the textbook fix for a dispatch-bound CG loop, and it was
implemented (`lssem3d/cupy_graph.py`) — but **CuPy cannot capture cuBLAS
calls** (`NotImplementedError: calling cuBLAS API during stream capture is
currently unsupported`), and the einsum contractions in the operator route
through `gemmStridedBatchedEx`. PyTorch *can* capture cuBLAS, so graphs remain
open on the torch port — a concrete advantage for it on hosts like this one.

The dispatch reductions above are the CuPy-side answer instead.

## Saving results

Colab VMs are ephemeral. Mount Drive before anything long, and write checkpoints
and diagnostics there rather than to `/content`.

In [ ]:
# 8. Optional: persist output.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lssem_results
!cp -v /content/lssem/scratch/*.npz /content/drive/MyDrive/lssem_results/ 2>/dev/null || echo 'no npz yet'